---
title: "Modelado y Pronóstico con LightGBM-Tweedie: Ruta Calle 72"
---

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Parámetros del experimento
N_TRIALS = 100            # ensayos máximos de Optuna por horizonte (5 para una prueba rápida)
WINDOW_MAX = 26000        # ventana máxima de la búsqueda = bloque de calentamiento
BLOQUE_DIAG = 1000        # franjas finales usadas en las pruebas de residuos
FORZAR_BUSQUEDA = False   # True: correr (o continuar) la búsqueda aunque ya exista su base de Optuna

# Horizontes: franjas de 15 min, minutos y carpeta de resultados de cada búsqueda
HORIZONTES = {
    2: {"minutos": 30,  "exec_id": 1},
    4: {"minutos": 60,  "exec_id": 2},
    8: {"minutos": 120, "exec_id": 3},
}

# Núcleos físicos del computador para LightGBM
try:
    import psutil
    N_THREADS = psutil.cpu_count(logical=False) or os.cpu_count()
except ImportError:
    N_THREADS = max(1, (os.cpu_count() or 2) // 2)
print(f"Núcleos lógicos: {os.cpu_count()} | núcleos que usará LightGBM: {N_THREADS}")

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm_cl72"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Partición 80/20 por franjas hecha en el EDA (03_exploratory_data_analysis_CL72)
train = pd.read_parquet(DATA_PROCESSED / "train_c72.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_c72.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print("Ruta:    Calle 72")
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Núcleos lógicos: 8 | núcleos que usará LightGBM: 4
Ruta:    Calle 72
Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


Las funciones siguientes concentran los pasos que se repiten en los tres horizontes: estimación de la potencia Tweedie, búsqueda de hiperparámetros, modelo final, diagnóstico de residuos, pronóstico del *holdout*, métricas y gráficas. Cada sección las llama con su horizonte $h$, de modo que los tres modelos se entrenan y evalúan exactamente con el mismo procedimiento.

In [2]:
def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }


def _pendiente_sin_recorte(serie: pd.Series) -> float:
    """Misma regresión que tt._estimate_tweedie_p_internal, sin el recorte a [1.01, 1.99]."""
    df = pd.DataFrame({"hora": serie.index.hour, "dow": serie.index.dayofweek, "y": serie.values})
    g = (df.groupby(["hora", "dow"])["y"].agg(["mean", "var", "count"])
           .query("count >= 30").dropna())
    g = g[(g["mean"] > 0) & (g["var"] > 0)]
    return float(np.polyfit(np.log(g["mean"]), np.log(g["var"]), 1)[0])


def estimar_potencia(h: int) -> float:
    """Potencia Tweedie del objetivo de h franjas, estimada solo con el bloque de calentamiento."""
    m = HORIZONTES[h]["minutos"]
    y_h = train["pasajeros"].iloc[:WINDOW_MAX].rolling(h).sum().shift(-(h - 1)).dropna()
    p = round(tt._estimate_tweedie_p_internal(y_h), 4)
    pendiente = _pendiente_sin_recorte(y_h)
    print(f"h = {h} ({m} min) | pendiente estimada: {pendiente:.4f} | potencia usada: p = {p}")
    if p >= 1.99 or p <= 1.01:
        print("  La pendiente quedó fuera de [1.01, 1.99]: se usa el tope del recorte, no un valor estimado.")
    tt.plot_tweedie_fit(
        series=y_h,
        p=p,
        titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
        xlab=f"Pasajeros acumulados en {m} min (h={h})",
        pads=(15, 15, 12),
        legend_bbox=(1.0, 0.95),
    )
    return p


def crear_trainer(h: int, p: float, exec_dir: Path) -> WalkForwardTrainer:
    return WalkForwardTrainer(
        model_class=LightGBMTweedieWrapper,
        scorer=DemandScorer(),
        output_dir=exec_dir,
        feature_builder=feature_builder,
        horizon_slots=h,
        n_splits=5,
        warmup=WINDOW_MAX,
        embargo=h - 1,
        n_trials=N_TRIALS,
        suggest_params_fn=lgbm_search_space,
        early_stop_rounds=30,
        inner_val_frac=0.1,
        model_kwargs={
            "tweedie_variance_power": p,
            "n_estimators": 5000,
            "early_stopping_rounds": 300,
            "num_threads": N_THREADS,
            "seed": SEED,
        },
    )


def buscar_hiperparametros(h: int, p: float):
    """Corre la búsqueda de Optuna, o la carga si ya existe su base (no repite ensayos)."""
    exec_dir = RESULTS_PATH / f"exec_{HORIZONTES[h]['exec_id']}"
    exec_dir.mkdir(parents=True, exist_ok=True)
    db = exec_dir / "optuna_study.db"
    if db.exists() and not FORZAR_BUSQUEDA:
        study = optuna.load_study(
            study_name=f"LightGBMTweedieWrapper_h{h}_study",
            storage=f"sqlite:///{db.resolve()}",
        )
        print(f"Búsqueda cargada de {exec_dir.name} ({len(study.trials)} ensayos); no se repite.")
        return study, exec_dir
    study = crear_trainer(h, p, exec_dir).run(train)
    return study, exec_dir


def resumen_busqueda(study, exec_dir: Path) -> None:
    estados = [t.state for t in study.trials]
    completos = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    df_csv = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")
    inicio = min(t.datetime_start for t in study.trials)
    fin = max(t.datetime_complete for t in study.trials if t.datetime_complete is not None)
    tabla = pd.DataFrame([{
        "Total Trials": len(study.trials),
        "Completos": len(completos),
        "Podados": estados.count(optuna.trial.TrialState.PRUNED),
        "Fallidos": estados.count(optuna.trial.TrialState.FAIL),
        "Mejor Trial": study.best_trial.number,
        "Mejor Tweedie Dev": round(study.best_value, 4),
        "Pliegues Entrenados": len(df_csv),
        "Tiempo Entrenamiento (min)": round(df_csv["train_time_sec"].sum() / 60, 2),
        "Tiempo por Pliegue (s)": round(df_csv["train_time_sec"].mean(), 3),
        "Duración Búsqueda (min)": round((fin - inicio).total_seconds() / 60, 2),
    }])
    tt.display_styled_df(df=tabla, decimals=4, align_left_first=False)


def top5_configuraciones(exec_dir: Path) -> None:
    df = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")
    df["lr"] = df["learning_rate"].apply(lambda x: f"{x:.4f}")
    df["l1"] = df["lambda_l1"].apply(lambda x: f"{x:.1e}")
    df["l2"] = df["lambda_l2"].apply(lambda x: f"{x:.1e}")
    cols = ["window", "num_leaves", "min_data_in_leaf", "feature_fraction",
            "bagging_fraction", "lr", "l1", "l2"]
    # Solo configuraciones con los 5 pliegues completos
    df = df[df.groupby("trial")["fold"].transform("nunique") == 5]
    agg = df.groupby(cols)["tweedie_deviance"].agg(["mean", "std"]).reset_index()
    agg["Tweedie Deviance"] = agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
    top = agg.sort_values("mean").head(5).rename(columns={
        "window": "Ventana", "num_leaves": "Hojas", "min_data_in_leaf": "Min Data Hoja",
        "feature_fraction": "Feat. Frac", "bagging_fraction": "Bagging Frac",
        "lr": "LR", "l1": "L1", "l2": "L2",
    })
    tt.display_styled_df(
        df=top[["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac",
                "LR", "L1", "L2", "Tweedie Deviance"]],
        decimals=4, align_left_first=False,
    )


def entrenar_modelo_final(h: int, p: float, study, exec_dir: Path) -> dict:
    """Entrena una vez con la mejor configuración sobre las últimas `window` franjas."""
    best = dict(study.best_params)
    for k, v in best.items():
        print(f"{k:>24}: {v}")
    window = best["window"]
    params = {k: v for k, v in best.items() if k != "window"}
    trainer = crear_trainer(h, p, exec_dir)
    X, y, y_naive, _ = trainer._build_dataset(train)
    modelo = LightGBMTweedieWrapper(
        **params,
        tweedie_variance_power=p,
        n_estimators=5000,
        early_stopping_rounds=300,
        inner_val_frac=0.3,
        num_threads=N_THREADS,
        seed=SEED,
    )
    modelo.fit(X.iloc[-window:], y[-window:])
    pred = modelo.predict_in_sample(X.iloc[-window:])
    print(f"\nVentana de entrenamiento: {X.index[-window]} -> {X.index[-1]} ({window} franjas)")
    return {"h": h, "p": p, "window": window, "trainer": trainer, "modelo": modelo,
            "X": X, "y": y, "y_naive": y_naive, "pred_train": pred}


def diagnostico_residuos(y_true, y_pred, h: int) -> None:
    """Ljung-Box y ARCH-LM sobre las últimas BLOQUE_DIAG franjas, tomando una de cada h
    (ventanas no solapadas)."""
    res = np.asarray(y_true) - np.asarray(y_pred)
    bloque = res[-BLOQUE_DIAG::h]
    z = (bloque - bloque.mean()) / bloque.std(ddof=1)
    diag = tt.run_model_diagnostics(z, lags=(10, 20, 30))
    m = HORIZONTES[h]["minutos"]
    print(f"{len(bloque)} ventanas no solapadas de {m} min | rezagos 10, 20 y 30 = "
          f"{10 * m / 60:g}, {20 * m / 60:g} y {30 * m / 60:g} horas de operación")
    tt.display_styled_df(df=diag, p_value_cols=["p-valor", "p-ajustado"], align_left_first=True)


def grafica_ajuste(fechas, y_true, y_pred, h: int, zooms: tuple, titulo: str) -> None:
    m = HORIZONTES[h]["minutos"]
    (zl, tl), (zr, tr) = zooms
    ts.plot_temporal_fit_panel(
        dates=fechas, y_true=y_true, y_pred=y_pred,
        zoom_left=zl, zoom_right=zr, text_left=tl, text_right=tr,
        title=titulo,
        xlab_left="Día del mes", xlab_right="Hora del día",
        ylab_top=f"Suma diaria de ventanas de {m} min",
        ylab_zoom=f"Pasajeros por ventana ({m} min)",
        zoom_ylim_pad=(0.10, 0.12), hspace=0.1, wspace=0.05,
    )


def pronosticar_holdout(final: dict) -> pd.DataFrame:
    """Pronóstico del holdout con reentrenamiento rodante por bloques (window fijo)."""
    h = final["h"]
    X_all, y_all, y_naive_all, _ = final["trainer"]._build_dataset(pd.concat([train, holdout]))
    mask = X_all.index >= holdout.index.min()
    t0 = time.time()
    pred = final["modelo"].forecast(
        X_seq=X_all.loc[mask], y_seq=y_all[mask],
        window=final["window"], embargo=h - 1, n_splits=5,
    )
    df = pd.DataFrame({"date": X_all.index[mask], "y_true": y_all[mask],
                       "y_naive": y_naive_all[mask], "y_pred": pred})
    ruta = RESULTS_PATH / f"lgbm_holdout_h{h}_{HORIZONTES[h]['minutos']}m_predictions.csv"
    df.to_csv(ruta, index=False)
    print(f"{len(df)} ventanas pronosticadas en {time.time() - t0:.1f} s -> {ruta.name}")
    return df


def comparar_metricas(final: dict, df_preds: pd.DataFrame) -> None:
    w = final["window"]
    ts.display_train_test_comparison(
        y_train=final["y"][-w:], preds_train=final["pred_train"],
        y_naive_train=final["y_naive"][-w:],
        y_test=df_preds["y_true"].to_numpy(), preds_test=df_preds["y_pred"].to_numpy(),
        y_naive_test=df_preds["y_naive"].to_numpy(),
        p=final["p"], error_relativo=True,
    )
    mae_m = (df_preds["y_true"] - df_preds["y_pred"]).abs().mean()
    mae_n = (df_preds["y_true"] - df_preds["y_naive"]).abs().mean()
    m = HORIZONTES[final["h"]]["minutos"]
    print(f"Holdout | demanda media: {df_preds['y_true'].mean():.1f} pasajeros por ventana de {m} min | "
          f"MAE modelo: {mae_m:.1f} | MAE ingenuo: {mae_n:.1f} | mejora: {100 * (1 - mae_m / mae_n):.1f} %")


ZOOMS_TRAIN = ((("2025-02-02", "2025-02-08"), "Febrero 2025"), (("2025-02-28", "2025-02-28"), "28 Febrero 2025"))
ZOOMS_HOLDOUT = ((("2025-09-01", "2025-09-07"), "Septiembre 2025"), (("2025-10-15", "2025-10-15"), "15 Octubre 2025"))
RESULTADOS = {}

# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie es un conteo de pasajeros cuya varianza crece con la media (ver la estimación de $p$ más adelante), estructura que una pérdida cuadrática o una verosimilitud Poisson no representan. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta Calle 72, con el mismo protocolo aplicado en la ruta El Silencio. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte. Para cada intervalo operativo solicitado (30, 60 y 120 minutos) se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante un reentrenamiento rodante por bloques con ventana deslizante. Este esquema respeta la partición cronológica y las restricciones metodológicas del proyecto: emplea exclusivamente variables disponibles en el instante de decisión y previene cualquier fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estima empíricamente a partir de la relación varianza-media de la serie. El valor resultante se fija como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

Se entrena un modelo LightGBM independiente por cada horizonte de predicción (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas). El *target* de cada modelo es la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo todos el mismo conjunto de 23 variables predictoras. Dado que la agregación temporal altera la dispersión de la demanda, el parámetro de potencia $p$ de Tweedie se estima empíricamente de forma independiente para cada horizonte y se fija como constante durante su entrenamiento. Cada intervalo se predice en un único paso, sin usar las predicciones de un horizonte como insumo de otro.

Para adaptar el modelo al nivel de demanda más reciente (en el análisis exploratorio, las subidas de un día hábil bajan un 13,1 % entre 2023 y 2025), se implementó una ventana deslizante (*sliding window*). El hiperparámetro `window` define la historia reciente que observa el modelo; se exploraron tamaños entre 18,000 y 26,000 franjas (de 8 a 12 meses), con un calentamiento inicial fijado en 26,000.

La validación en `train_c72.parquet` usa un esquema *walk-forward* de 5 pliegues, aplicando un *embargo* de $h-1$ franjas en la frontera para evitar fugas de información. Para la evaluación final, el `holdout_c72.parquet` replica esta mecánica avanzando cronológicamente y reentrenando el modelo por bloques con las últimas `window` franjas disponibles.

La implementación utiliza `LightGBM` (v4.6) con el objetivo `tweedie`. El *early stopping* interno se evalúa sobre el último tramo cronológico de cada bloque de entrenamiento (10 % en la búsqueda de hiperparámetros y 30 % en el modelo final), respetando también el *embargo* del horizonte.

Para los diagnósticos de residuos se usa el último bloque de 1.000 franjas de cada conjunto, tomando una franja de cada $h$ para que las ventanas evaluadas no se solapen (500 ventanas en 30 minutos, 250 en 60 y 125 en 120). Así, el rezago 10 equivale a 5, 10 y 20 horas de operación en cada horizonte.

En las gráficas de ajuste, el panel superior suma por día el objetivo de cada franja. Como cada franja aparece en $h$ ventanas consecutivas, esos totales equivalen aproximadamente a 2, 4 y 8 veces los pasajeros reales del día en los horizontes de 30, 60 y 120 minutos; los paneles inferiores sí muestran pasajeros por ventana.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para determinar el parámetro de potencia $p$ en la muestra de entrenamiento, se aplicó un estimador por momentos basado en la relación teórica entre la varianza y la media de la distribución Tweedie:

$$
\log(\operatorname{Var}(y)) = \log(\phi) + p \log(\mu)
$$

Las observaciones del bloque de calentamiento (las primeras 26.000 franjas del entrenamiento) se agrupan por hora y día de la semana, y $p$ es la pendiente de la regresión de $\log(\operatorname{Var})$ sobre $\log(\mu)$ en esos grupos. Se usa la misma función que en la ruta El Silencio (`_estimate_tweedie_p_internal` de `trainer_toolkit`), y solo el bloque de calentamiento, para que el valor no use información de los bloques de validación. La gráfica compara la distribución observada del objetivo con la distribución Tweedie ajustada y su gráfico Q-Q.

In [3]:
P2 = estimar_potencia(2)

h = 2 (30 min) | pendiente estimada: 1.1122 | potencia usada: p = 1.1122


Con el bloque de calentamiento (las primeras 26.000 franjas, del 1 de enero de 2023 al 4 de febrero de 2024) se obtuvo $p = 1.1122$. El valor está cerca de 1: en Calle 72 la varianza del objetivo de 30 minutos crece poco más rápido que la media, y la distribución está más cerca de una Poisson que de una Gamma.

En la gráfica, la curva Tweedie sigue la forma del histograma en la zona central, entre unos 100 y 400 pasajeros cada 30 minutos. Las diferencias están en los extremos: los datos tienen más franjas con cero pasajeros de las que asigna la distribución ajustada (primera barra del histograma), y en el gráfico Q-Q los cuantiles observados quedan por encima de la recta a partir de unos 500 pasajeros, es decir, la cola derecha de los datos es más pesada que la de la distribución ajustada.

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de 5 pliegues sobre el conjunto de entrenamiento. La búsqueda de Optuna exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El modelo recibe directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al nivel de demanda más reciente, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se reentrena con un bloque de historia delimitado por el hiperparámetro `window` y se valida con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de datos y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`) y las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, la cual se evalúa sobre el último 10% cronológico de cada bloque de entrenamiento, respetando también el embargo de una franja. El desempeño se evaluó mediante la desviación Tweedie, fijando el parámetro de potencia en el valor estimado en la celda anterior (calculado solo con el bloque de calentamiento para evitar fuga de información) y aplicando un límite superior de 100 por observación para mitigar el efecto de eventos atípicos impredecibles.

In [4]:
study_h2, exec_dir_h2 = buscar_hiperparametros(2, P2)

Búsqueda cargada de exec_1 (81 ensayos); no se repite.


La búsqueda realizó 81 de los 100 ensayos programados (numerados del 0 al 80). La mejor configuración se encontró en el ensayo 50 y los 30 siguientes no la mejoraron, lo que activó la parada temprana. El mecanismo de poda descartó 19 configuraciones antes de completar sus 5 pliegues y ninguna falló, por lo que 62 configuraciones completaron la validación. La menor desviación Tweedie promedio fue 15,026. Se entrenaron 380 pliegues, incluidos los de las configuraciones podadas; su entrenamiento sumó 16,0 minutos (2,5 segundos por pliegue) con LightGBM usando 10 núcleos, y la búsqueda completa duró 19,1 minutos.

In [5]:
resumen_busqueda(study_h2, exec_dir_h2)

Total Trials,Completos,Podados,Fallidos,Mejor Trial,Mejor Tweedie Dev,Pliegues Entrenados,Tiempo Entrenamiento (min),Tiempo por Pliegue (s),Duración Búsqueda (min)
81,62,19,0,50,15.0256,380,16.0300,2.5310,19.1400


Las cinco mejores configuraciones tienen una desviación Tweedie promedio entre 15,026 y 15,036, con una desviación estándar entre pliegues de 2,13 a 2,23. La diferencia entre ellas (0,01) es mucho menor que la variabilidad entre pliegues, por lo que en la práctica empatan. Las cinco eligieron la misma ventana de historia, 22.000 franjas (unos 338 días, poco más de 11 meses), y comparten una fracción de variables baja (0,40 a 0,45), una fracción de filas de 0,45 a 0,50 y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero (menores a $10^{-3}$). Varían en el número de hojas (29 a 69), el mínimo de datos por hoja (101 a 182) y la tasa de aprendizaje (0,007 a 0,022).

La tabla solo incluye configuraciones que completaron los 5 pliegues.

In [6]:
top5_configuraciones(exec_dir_h2)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"22,000",29,109,0.4000,0.4500,0.0221,9.0e-08,2.0e-05,15.026 ± 2.154
"22,000",50,128,0.4000,0.4500,0.0069,8.0e-07,8.4e-05,15.026 ± 2.132
"22,000",69,182,0.4000,0.5000,0.0193,2.8e-07,5.7e-04,15.030 ± 2.185
"22,000",46,104,0.4500,0.4500,0.0205,6.4e-08,1.4e-05,15.033 ± 2.190
"22,000",42,101,0.4000,0.4500,0.0153,2.9e-08,6.0e-04,15.036 ± 2.229


El modelo final usa la configuración con menor desviación promedio (`study.best_params`, primera fila de la tabla): ventana de 22.000 franjas, 29 hojas, tasa de aprendizaje de 0,022, mínimo de 109 datos por hoja, fracción de variables de 0,40 y fracción de filas de 0,45. La segunda configuración tiene el mismo promedio con tres decimales y una variabilidad entre pliegues ligeramente menor (2,132 frente a 2,154); la diferencia entre ambas no es relevante. El modelo se entrena una sola vez con las últimas 22.000 franjas del conjunto de entrenamiento, del 22 de junio de 2024 al 26 de mayo de 2025.

In [7]:
final_h2 = entrenar_modelo_final(2, P2, study_h2, exec_dir_h2)

                  window: 22000
              num_leaves: 29
           learning_rate: 0.02214065095014879
        min_data_in_leaf: 109
        feature_fraction: 0.4
        bagging_fraction: 0.45
            bagging_freq: 2
               lambda_l1: 9.028957269769347e-08
               lambda_l2: 2.046708015812977e-05
 min_sum_hessian_in_leaf: 1.1365415517056803



Ventana de entrenamiento: 2024-06-22 09:45:00 -> 2025-05-26 17:00:00 (22000 franjas)


Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento (`window` franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Por eso el diagnóstico se realiza sobre el último bloque de 1.000 franjas, tomando una de cada dos para que las ventanas de 30 minutos no se solapen (500 ventanas). La corrección de Holm ajusta los p-valores por las tres pruebas de cada tipo.

In [8]:
diagnostico_residuos(final_h2["y"][-final_h2["window"]:], final_h2["pred_train"], 2)

500 ventanas no solapadas de 30 min | rezagos 10, 20 y 30 = 5, 10 y 15 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),8.552,.575,.837,Sin Autocorrelación
Ljung-Box (Lag 20),24.392,.226,.677,Sin Autocorrelación
Ljung-Box (Lag 30),30.939,.418,.837,Sin Autocorrelación
ARCH-LM (Lag 10),28.450,.002,.005,Efectos ARCH Significativos
ARCH-LM (Lag 20),34.708,.022,.043,Efectos ARCH Significativos
ARCH-LM (Lag 30),26.333,.658,.658,Sin Efectos ARCH


Con las 500 ventanas no solapadas, las pruebas de Ljung-Box no encontraron evidencia de autocorrelación en ninguno de los rezagos evaluados ($p_{\text{ajustado}} \ge 0{,}677$). Las pruebas ARCH-LM rechazan en los rezagos 10 y 20 ($p_{\text{ajustado}} = 0{,}005$ y $0{,}043$) y no en el rezago 30 ($p_{\text{ajustado}} = 0{,}658$): la dispersión del error cambia en el tiempo, lo esperable cuando la varianza de la demanda crece con la media.

La gráfica compara la demanda observada con la predicha en la ventana de entrenamiento. Los paneles inferiores muestran por franja la semana del 2 al 8 de febrero de 2025 y el 28 de febrero de 2025.

In [9]:
w = final_h2["window"]
grafica_ajuste(
    final_h2["X"].index[-w:], final_h2["y"][-w:], final_h2["pred_train"], 2,
    ZOOMS_TRAIN, "Demanda acumulada de pasajeros 2024-2025 (30 min)",
)

El modelo sigue el ciclo semanal en toda la ventana de entrenamiento: niveles altos en días hábiles, intermedios los sábados y bajos los domingos. Las mayores diferencias se ven en días puntuales con caídas fuertes, como el 1 de enero de 2025, donde la demanda observada queda por debajo de la predicha. En los acercamientos por franja, el modelo reproduce la forma del día pero suaviza los picos: el 28 de febrero de 2025 predice cerca de 490 pasajeros en el pico de la mañana frente a unos 650 observados, y no reproduce el pico de las 13:00 (unos 550 observados).

## Evaluación en el holdout

La evaluación del rendimiento del modelo LightGBM-Tweedie requiere contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la ventana activa de entrenamiento (*In-Sample*, `window` franjas) permite verificar la convergencia del algoritmo a la dinámica histórica de operación, la evaluación fuera de muestra (*Out-of-Sample*, conjunto *holdout*) determina la utilidad operativa real del modelo ante nuevos regímenes de movilidad.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basan en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con la potencia estimada). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

No obstante, en el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debe considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afecta de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window` elegido en la búsqueda, `embargo` = $h - 1$), generando pronósticos puntuales en pasajeros acumulados por ventana.

In [10]:
preds_h2 = pronosticar_holdout(final_h2)
RESULTADOS[2] = preds_h2

14247 ventanas pronosticadas en 5.4 s -> lgbm_holdout_h2_30m_predictions.csv


Diagnóstico de los residuos fuera de muestra: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas del *holdout*, tomando una de cada dos (500 ventanas no solapadas de 30 minutos).

A diferencia del entrenamiento, fuera de muestra ambas pruebas rechazan en los tres rezagos ($p < .001$): el estadístico de Ljung-Box está entre 117,7 y 181,8 y el de ARCH-LM entre 59,6 y 72,9. El error del *holdout* conserva dependencia entre ventanas consecutivas, lo que es consistente con días completos en los que el modelo se equivoca en la misma dirección (ver la gráfica de ajuste del *holdout*).

In [11]:
diagnostico_residuos(preds_h2["y_true"], preds_h2["y_pred"], 2)

500 ventanas no solapadas de 30 min | rezagos 10, 20 y 30 = 5, 10 y 15 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),117.703,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),165.587,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),181.835,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),59.638,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),71.283,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),72.905,< .001,< .001,Efectos ARCH Significativos


Comparación de métricas entre la ventana de entrenamiento y el *holdout*, junto con el modelo ingenuo semanal (misma franja de la semana anterior). Según el protocolo, una variación relativa de la desviación Tweedie mayor al 5 % en valor absoluto, en cualquier dirección, sale de la tolerancia y debe explicarse.

La desviación Tweedie baja un 0,68 % del entrenamiento al *holdout* (de 11,976 a 11,895), dentro de la tolerancia, por lo que no hay evidencia de sobreajuste. El MAE (50,8 y 50,3), el RMSE (67,1 y 67,0) y el SMAPE (0,284 y 0,287) prácticamente no cambian. El MASE sube un 7,5 % (de 0,637 a 0,685), pero sigue por debajo de 1 en ambos periodos, es decir, el modelo se equivoca menos que el modelo ingenuo: en el *holdout*, su MAE es de 50,3 pasajeros por cada 30 minutos frente a 73,4 del ingenuo (31,5 % menos), con una demanda media de 206 pasajeros. El sesgo pasa de −1,1 en entrenamiento a +11,0 en el *holdout*: fuera de muestra el modelo sobreestima en promedio 11 pasajeros por cada 30 minutos (5 % de la demanda media). El error relativo de +1.090 % del sesgo se debe a que el valor de entrenamiento es cercano a cero.

In [12]:
comparar_metricas(final_h2, preds_h2)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,11.9760,11.8947,-0.6791%
MAE,50.8383,50.2733,-1.1115%
RMSE,67.1367,67.0174,-0.1778%
MSE,4507.3387,4491.3286,-0.3552%
Bias,-1.1123,11.0127,+1090.1159%
SMAPE,0.2837,0.2872,+1.2359%
MASE,0.6374,0.6852,+7.5130%
Pinball (τ=0.50),25.4192,25.1366,-1.1115%


Holdout | demanda media: 206.0 pasajeros por ventana de 30 min | MAE modelo: 50.3 | MAE ingenuo: 73.4 | mejora: 31.5 %


Ajuste en el *holdout*: demanda observada frente a predicha, por día y en dos acercamientos por franja. El primer punto es bajo porque el *holdout* empieza el 26 de mayo de 2025 a las 17:30 y ese día solo tiene 13 franjas.

La sobreestimación no es pareja en el periodo: el sesgo medio es del 10 % en junio, 9 % en julio y 7 % en agosto, baja al 5 % en septiembre, al 1,5 % en octubre y al 0,1 % en noviembre, y sube al 5 % en diciembre. Los días con mayor sobreestimación son días con demanda atípicamente baja: el 19 de junio, el 9 y el 29 de julio y el 24 de diciembre (señalados como atípicos en el análisis exploratorio), el 25 de diciembre y el 31 de diciembre de 2025; en ellos la predicción supera a lo observado entre un 38 % y un 91 %. En los acercamientos por franja, el modelo sigue la forma de los días hábiles de la primera semana de septiembre y, como en entrenamiento, suaviza los picos: el 15 de octubre predice cerca de 520 pasajeros en el pico de la mañana frente a unos 790 observados.

In [13]:
grafica_ajuste(
    pd.to_datetime(preds_h2["date"]), preds_h2["y_true"].to_numpy(), preds_h2["y_pred"].to_numpy(), 2,
    ZOOMS_HOLDOUT, "Demanda acumulada de pasajeros Holdout 2025 (30 min)",
)

## Ventana de predicción de 60 minutos

### Entrenamiento

La potencia Tweedie del objetivo de 60 minutos ($h = 4$) se estima con el mismo procedimiento del horizonte de 30 minutos: regresión de $\log(\operatorname{Var})$ sobre $\log(\mu)$ por hora y día de la semana, solo con el bloque de calentamiento. La celda imprime la pendiente estimada y la potencia usada; si la pendiente queda fuera de $[1.01, 1.99]$, la función la recorta al tope y el valor usado no es una estimación.

In [14]:
P4 = estimar_potencia(4)

h = 4 (60 min) | pendiente estimada: 1.5227 | potencia usada: p = 1.5227


Con el bloque de calentamiento se obtuvo $p = 1{,}5227$, un valor estimado y no recortado. Es mayor que el de 30 minutos ($p = 1{,}1122$): al acumular más franjas, la varianza crece más rápido con la media y la distribución se aleja de una Poisson.

En la gráfica, la curva Tweedie sigue el cuerpo del histograma, aunque los datos muestran dos modas (cerca de 280 y de 500 pasajeros por hora) que la curva ajustada suaviza en una sola. Los datos tienen más ventanas con cero pasajeros que las que asigna la distribución, y en el gráfico Q-Q los cuantiles observados quedan por debajo de la recta a partir de unos 1.350 pasajeros: la cola derecha de los datos es más liviana que la de la distribución ajustada.

Para la selección de hiperparámetros del horizonte de 60 minutos ($h = 4$ franjas) se replica el protocolo del horizonte de 30 minutos: validación *walk-forward* de cinco pliegues con ventana deslizante, el mismo espacio de búsqueda en Optuna, parada temprana tras 30 ensayos sin mejora y la potencia Tweedie estimada en la celda anterior. El *embargo* entre entrenamiento y validación es de $h - 1 = 3$ franjas, para que las ventanas acumuladas de 60 minutos no se solapen entre bloques.

In [15]:
study_h4, exec_dir_h4 = buscar_hiperparametros(4, P4)

  0%|          | 0/100 [00:00<?, ?it/s]

La búsqueda completó los 100 ensayos programados sin activar la parada temprana: la mejor configuración se encontró en el ensayo 89, y no alcanzaron a pasar 30 ensayos sin mejora antes del límite. El mecanismo de poda descartó 5 configuraciones y ninguna falló, por lo que 95 completaron la validación. La menor desviación Tweedie promedio fue 1,8219. Se entrenaron 491 pliegues, con un promedio de 9,3 segundos por pliegue; el entrenamiento sumó 76,3 minutos y la búsqueda completa duró 88,3 minutos.

In [16]:
resumen_busqueda(study_h4, exec_dir_h4)

Total Trials,Completos,Podados,Fallidos,Mejor Trial,Mejor Tweedie Dev,Pliegues Entrenados,Tiempo Entrenamiento (min),Tiempo por Pliegue (s),Duración Búsqueda (min)
100,95,5,0,89,1.8219,491,76.3000,9.3240,88.2700


Las cinco mejores configuraciones tienen una desviación Tweedie promedio entre 1,822 y 1,827, con una desviación estándar entre pliegues cercana a 0,71; la diferencia entre ellas es mucho menor que esa variabilidad, por lo que en la práctica empatan. Las cinco eligieron la ventana de 26.000 franjas, que es el máximo del espacio de búsqueda, lo que indica que conviene probar ventanas más largas en este horizonte. Comparten una fracción de variables de 0,40, una fracción de filas de 0,40 a 0,45, una tasa de aprendizaje baja (0,0010 a 0,0011) y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero, con árboles grandes (195 a 277 hojas).

In [17]:
top5_configuraciones(exec_dir_h4)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",195,118,0.4000,0.4500,0.0011,3.8e-05,2.5e-04,1.822 ± 0.709
"26,000",277,55,0.4000,0.4500,0.0010,2.6e-06,5.1e-05,1.824 ± 0.702
"26,000",260,92,0.4000,0.4000,0.0011,1.3e-05,2.4e-03,1.826 ± 0.706
"26,000",236,93,0.4000,0.4500,0.0011,1.3e-05,6.2e-05,1.827 ± 0.709
"26,000",258,87,0.4000,0.4000,0.0011,1.0e-05,1.6e-05,1.827 ± 0.709


El modelo final usa la configuración con menor desviación promedio (`study.best_params`, primera fila de la tabla): ventana de 26.000 franjas, 195 hojas, tasa de aprendizaje de 0,0011, mínimo de 118 datos por hoja, fracción de variables de 0,40 y fracción de filas de 0,45. Se entrena una sola vez con las últimas 26.000 franjas del conjunto de entrenamiento, del 21 de abril de 2024 al 26 de mayo de 2025.

In [18]:
final_h4 = entrenar_modelo_final(4, P4, study_h4, exec_dir_h4)

                  window: 26000
              num_leaves: 195
           learning_rate: 0.0011184132017428568
        min_data_in_leaf: 118
        feature_fraction: 0.4
        bagging_fraction: 0.45
            bagging_freq: 6
               lambda_l1: 3.7610069659355765e-05
               lambda_l2: 0.00024545771623278644
 min_sum_hessian_in_leaf: 1.8432182994708688



Ventana de entrenamiento: 2024-04-21 16:45:00 -> 2025-05-26 16:30:00 (26000 franjas)


Diagnóstico de los residuos de entrenamiento: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas de la ventana de entrenamiento, tomando una de cada 4 (250 ventanas no solapadas de 60 minutos).

Las pruebas de Ljung-Box no encontraron evidencia de autocorrelación ($p_{\text{ajustado}} \ge 0{,}115$). Las pruebas ARCH-LM no rechazan en el rezago 10 ($p_{\text{ajustado}} = 0{,}984$), pero sí en los rezagos 20 y 30 ($p_{\text{ajustado}} < .001$): la dispersión del error cambia entre días, aunque no dentro de las primeras 10 horas.

In [19]:
diagnostico_residuos(final_h4["y"][-final_h4["window"]:], final_h4["pred_train"], 4)

250 ventanas no solapadas de 60 min | rezagos 10, 20 y 30 = 10, 20 y 30 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),11.141,.347,.347,Sin Autocorrelación
Ljung-Box (Lag 20),32.477,.038,.115,Sin Autocorrelación
Ljung-Box (Lag 30),40.399,.097,.195,Sin Autocorrelación
ARCH-LM (Lag 10),2.882,.984,.984,Sin Efectos ARCH
ARCH-LM (Lag 20),86.763,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),63.229,< .001,< .001,Efectos ARCH Significativos


La gráfica compara la demanda observada con la predicha en la ventana de entrenamiento. Los totales diarios del panel superior equivalen aproximadamente a 4 veces los pasajeros del día; los paneles inferiores muestran pasajeros por ventana de 60 minutos en la semana del 2 al 8 de febrero de 2025 y el 28 de febrero de 2025.

El modelo sigue el ciclo semanal durante toda la ventana. Entre febrero y abril de 2025, la demanda observada de los días hábiles queda por encima de la predicha. En los acercamientos, el modelo reproduce la forma del día pero queda por debajo de los picos: el 28 de febrero predice cerca de 620 pasajeros en el pico de la mañana frente a unos 1.170 observados, y cerca de 420 frente a unos 890 en el pico de las 13:00.

In [20]:
w = final_h4["window"]
grafica_ajuste(
    final_h4["X"].index[-w:], final_h4["y"][-w:], final_h4["pred_train"], 4,
    ZOOMS_TRAIN, "Demanda acumulada de pasajeros 2024-2025 (60 min)",
)

### Evaluación

Para el horizonte de 60 minutos se aplican el mismo protocolo de evaluación y los mismos criterios del horizonte de 30 minutos. Las predicciones del *holdout* se generan con reentrenamiento rodante por bloques (`n_splits = 5`, `window` elegido en la búsqueda) y un *embargo* de 3 franjas.

In [21]:
preds_h4 = pronosticar_holdout(final_h4)
RESULTADOS[4] = preds_h4

14245 ventanas pronosticadas en 30.2 s -> lgbm_holdout_h4_60m_predictions.csv


Diagnóstico de los residuos fuera de muestra: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas del *holdout*, tomando una de cada 4 (250 ventanas no solapadas de 60 minutos).

Fuera de muestra, las pruebas de Ljung-Box rechazan en los tres rezagos ($p < .001$, estadísticos entre 102,3 y 135,9). Las pruebas ARCH-LM rechazan en los rezagos 10 y 20 ($p_{\text{ajustado}} < .001$ y $0{,}006$) y no en el 30 ($p_{\text{ajustado}} = 0{,}053$). Como en 30 minutos, el error del *holdout* conserva dependencia entre ventanas consecutivas.

In [22]:
diagnostico_residuos(preds_h4["y_true"], preds_h4["y_pred"], 4)

250 ventanas no solapadas de 60 min | rezagos 10, 20 y 30 = 10, 20 y 30 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),102.322,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),124.382,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),135.876,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),34.311,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),41.972,.003,.006,Efectos ARCH Significativos
ARCH-LM (Lag 30),43.521,.053,.053,Sin Efectos ARCH


Comparación de métricas entre la ventana de entrenamiento y el *holdout*, junto con el modelo ingenuo semanal. Según el protocolo, una variación relativa de la desviación Tweedie mayor al 5 % en valor absoluto, en cualquier dirección, sale de la tolerancia y debe explicarse.

La desviación Tweedie baja un 8,25 % del entrenamiento al *holdout* (de 1,4173 a 1,3003), fuera de la tolerancia del 5 %. El cambio es en la dirección de un menor error, y las métricas de escala bajan menos: el MAE un 3,7 % (de 73,0 a 70,3) y el RMSE un 4,3 % (de 100,5 a 96,1). Una explicación consistente con la gráfica de entrenamiento es que la ventana de entrenamiento incluye febrero a abril de 2025, cuando el modelo queda por debajo del nivel de los días hábiles, lo que aumenta el error de entrenamiento; el resultado no indica sobreajuste, pero sale del criterio y queda señalado. El MASE sube un 8,0 % (de 0,624 a 0,674) y sigue por debajo de 1: en el *holdout*, el MAE del modelo es de 70,3 pasajeros por hora frente a 104,4 del ingenuo (32,6 % menos), con una demanda media de 412 pasajeros. El sesgo pasa de −11,6 en entrenamiento (subestima) a +17,7 en el *holdout* (sobreestima un 4,3 % de la demanda media).

In [23]:
comparar_metricas(final_h4, preds_h4)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,1.4173,1.3003,-8.2526%
MAE,73.0168,70.3278,-3.6827%
RMSE,100.4643,96.1456,-4.2987%
MSE,10093.0788,9243.9801,-8.4127%
Bias,-11.5592,17.7292,+253.3772%
SMAPE,0.1990,0.1930,-3.0373%
MASE,0.6237,0.6737,+8.0151%
Pinball (τ=0.50),36.5084,35.1639,-3.6827%


Holdout | demanda media: 412.1 pasajeros por ventana de 60 min | MAE modelo: 70.3 | MAE ingenuo: 104.4 | mejora: 32.6 %


Ajuste en el *holdout*: demanda observada frente a predicha. Los totales diarios del panel superior equivalen aproximadamente a 4 veces los pasajeros del día; los paneles inferiores muestran pasajeros por ventana de 60 minutos en la primera semana de septiembre y el 15 de octubre de 2025.

El sesgo medio cambia de signo en el periodo: en junio el modelo subestima un 3,8 %, y de julio a diciembre sobreestima entre un 3,9 % y un 6,7 % por mes. Los días con mayor sobreestimación son el 31 de diciembre (88 %), el 25 de diciembre (58 %), el 24 de diciembre (53 %), el 19 de junio (36 %), el 29 de julio (33 %) y el puente festivo del 7 y 8 de diciembre (30 % y 28 %). En los acercamientos, el modelo sigue la forma de los días hábiles de la primera semana de septiembre; el 15 de octubre predice cerca de 1.070 pasajeros en el pico de la mañana frente a unos 1.300 observados.

In [24]:
grafica_ajuste(
    pd.to_datetime(preds_h4["date"]), preds_h4["y_true"].to_numpy(), preds_h4["y_pred"].to_numpy(), 4,
    ZOOMS_HOLDOUT, "Demanda acumulada de pasajeros Holdout 2025 (60 min)",
)

## Ventana de predicción de 120 minutos

### Entrenamiento

La potencia Tweedie del objetivo de 120 minutos ($h = 8$) se estima con el mismo procedimiento del horizonte de 30 minutos: regresión de $\log(\operatorname{Var})$ sobre $\log(\mu)$ por hora y día de la semana, solo con el bloque de calentamiento. La celda imprime la pendiente estimada y la potencia usada; si la pendiente queda fuera de $[1.01, 1.99]$, la función la recorta al tope y el valor usado no es una estimación.

In [25]:
P8 = estimar_potencia(8)

h = 8 (120 min) | pendiente estimada: 1.8448 | potencia usada: p = 1.8448


Con el bloque de calentamiento se obtuvo $p = 1{,}8448$, un valor estimado y no recortado. Es el mayor de los tres horizontes: al acumular 8 franjas, la distribución se acerca a una Gamma.

En la gráfica, los datos muestran dos modas (cerca de 550 y de 1.050 pasajeros cada dos horas) que la curva Tweedie suaviza en una sola, ubicada entre ambas. En el gráfico Q-Q los cuantiles observados siguen la recta en la zona central y quedan por debajo a partir de unos 1.900 pasajeros: la cola derecha de los datos es más liviana que la de la distribución ajustada.

Para la selección de hiperparámetros del horizonte de 120 minutos ($h = 8$ franjas) se replica el protocolo del horizonte de 30 minutos: validación *walk-forward* de cinco pliegues con ventana deslizante, el mismo espacio de búsqueda en Optuna, parada temprana tras 30 ensayos sin mejora y la potencia Tweedie estimada en la celda anterior. El *embargo* entre entrenamiento y validación es de $h - 1 = 7$ franjas, para que las ventanas acumuladas de 120 minutos no se solapen entre bloques.

In [26]:
study_h8, exec_dir_h8 = buscar_hiperparametros(8, P8)

  0%|          | 0/100 [00:00<?, ?it/s]

La búsqueda completó los 100 ensayos programados: la mejor configuración se encontró en el ensayo 72 y no alcanzaron a pasar 30 ensayos sin mejora antes del límite. El mecanismo de poda descartó 3 configuraciones y ninguna falló, por lo que 97 completaron la validación. La menor desviación Tweedie promedio fue 0,2456. Se entrenaron 495 pliegues, con un promedio de 1,7 segundos por pliegue; el entrenamiento sumó 14,0 minutos y la búsqueda completa duró 16,7 minutos.

In [27]:
resumen_busqueda(study_h8, exec_dir_h8)

Total Trials,Completos,Podados,Fallidos,Mejor Trial,Mejor Tweedie Dev,Pliegues Entrenados,Tiempo Entrenamiento (min),Tiempo por Pliegue (s),Duración Búsqueda (min)
100,97,3,0,72,0.2456,495,13.9600,1.6920,16.7300


Las cinco mejores configuraciones tienen una desviación Tweedie promedio entre 0,246 y 0,247, con una desviación estándar entre pliegues de 0,168 a 0,171, por lo que en la práctica empatan. Las cinco eligieron la ventana de 22.000 franjas, la misma que en 30 minutos, y comparten una fracción de variables baja (0,40 a 0,45), una fracción de filas alta (0,80 a 0,95) y árboles grandes (206 a 289 hojas), con tasas de aprendizaje de 0,014 a 0,047 y penalizaciones $\ell_1$ y $\ell_2$ bajas.

In [28]:
top5_configuraciones(exec_dir_h8)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"22,000",206,93,0.4000,0.8500,0.0140,4.7e-06,1.3e-03,0.246 ± 0.168
"22,000",289,101,0.4000,0.9000,0.0189,6.6e-06,6.8e-03,0.246 ± 0.168
"22,000",252,86,0.4500,0.8000,0.0470,3.3e-06,2.5e-03,0.247 ± 0.171
"22,000",265,95,0.4000,0.9500,0.0284,3.5e-05,5.2e-04,0.247 ± 0.168
"22,000",229,82,0.4500,0.8000,0.0311,8.0e-06,2.0e-02,0.247 ± 0.168


El modelo final usa la configuración con menor desviación promedio (`study.best_params`, primera fila de la tabla): ventana de 22.000 franjas, 206 hojas, tasa de aprendizaje de 0,014, mínimo de 93 datos por hoja, fracción de variables de 0,40 y fracción de filas de 0,85. Se entrena una sola vez con las últimas 22.000 franjas del conjunto de entrenamiento, del 22 de junio de 2024 al 26 de mayo de 2025.

In [29]:
final_h8 = entrenar_modelo_final(8, P8, study_h8, exec_dir_h8)

                  window: 22000
              num_leaves: 206
           learning_rate: 0.013990551653528895
        min_data_in_leaf: 93
        feature_fraction: 0.4
        bagging_fraction: 0.8500000000000001
            bagging_freq: 2
               lambda_l1: 4.668103450712864e-06
               lambda_l2: 0.0012545085291443357
 min_sum_hessian_in_leaf: 0.002028280636159267



Ventana de entrenamiento: 2024-06-22 08:15:00 -> 2025-05-26 15:30:00 (22000 franjas)


Diagnóstico de los residuos de entrenamiento: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas de la ventana de entrenamiento, tomando una de cada 8 (125 ventanas no solapadas de 120 minutos).

Ninguna de las pruebas rechaza: no hay evidencia de autocorrelación ($p_{\text{ajustado}} \ge 0{,}436$) ni de efectos ARCH ($p_{\text{ajustado}} = 1{,}000$) en la ventana de entrenamiento.

In [30]:
diagnostico_residuos(final_h8["y"][-final_h8["window"]:], final_h8["pred_train"], 8)

125 ventanas no solapadas de 120 min | rezagos 10, 20 y 30 = 20, 40 y 60 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),12.392,.260,.519,Sin Autocorrelación
Ljung-Box (Lag 20),26.652,.145,.436,Sin Autocorrelación
Ljung-Box (Lag 30),33.088,.319,.519,Sin Autocorrelación
ARCH-LM (Lag 10),11.068,.352,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),14.257,.817,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),19.261,.934,1.000,Sin Efectos ARCH


La gráfica compara la demanda observada con la predicha en la ventana de entrenamiento. Los totales diarios del panel superior equivalen aproximadamente a 8 veces los pasajeros del día; los paneles inferiores muestran pasajeros por ventana de 120 minutos en la semana del 2 al 8 de febrero de 2025 y el 28 de febrero de 2025.

El modelo sigue de cerca la demanda diaria durante toda la ventana, con las mayores diferencias en algunos días hábiles de febrero y marzo de 2025, cuando la demanda observada queda por encima de la predicha. En la semana del 2 al 8 de febrero, la predicción por ventana se ajusta a la observada; el 28 de febrero predice cerca de 1.100 pasajeros en el pico de la mañana frente a unos 1.930 observados.

In [31]:
w = final_h8["window"]
grafica_ajuste(
    final_h8["X"].index[-w:], final_h8["y"][-w:], final_h8["pred_train"], 8,
    ZOOMS_TRAIN, "Demanda acumulada de pasajeros 2024-2025 (120 min)",
)

### Evaluación

Para el horizonte de 120 minutos se aplican el mismo protocolo de evaluación y los mismos criterios del horizonte de 30 minutos. Las predicciones del *holdout* se generan con reentrenamiento rodante por bloques (`n_splits = 5`, `window` elegido en la búsqueda) y un *embargo* de 7 franjas.

In [32]:
preds_h8 = pronosticar_holdout(final_h8)
RESULTADOS[8] = preds_h8

14241 ventanas pronosticadas en 6.1 s -> lgbm_holdout_h8_120m_predictions.csv


Diagnóstico de los residuos fuera de muestra: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas del *holdout*, tomando una de cada 8 (125 ventanas no solapadas de 120 minutos).

Fuera de muestra, las pruebas de Ljung-Box rechazan en los tres rezagos ($p < .001$, estadísticos entre 49,1 y 63,9), mientras que las pruebas ARCH-LM no rechazan ($p_{\text{ajustado}} \ge 0{,}581$). A diferencia del entrenamiento, el error del *holdout* conserva dependencia entre ventanas consecutivas.

In [33]:
diagnostico_residuos(preds_h8["y_true"], preds_h8["y_pred"], 8)

125 ventanas no solapadas de 120 min | rezagos 10, 20 y 30 = 20, 40 y 60 horas de operación


Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),49.082,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),51.619,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),63.863,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),13.566,.194,.581,Sin Efectos ARCH
ARCH-LM (Lag 20),14.890,.783,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),16.184,.981,1.000,Sin Efectos ARCH


Comparación de métricas entre la ventana de entrenamiento y el *holdout*, junto con el modelo ingenuo semanal. Según el protocolo, una variación relativa de la desviación Tweedie mayor al 5 % en valor absoluto, en cualquier dirección, sale de la tolerancia y debe explicarse.

La desviación Tweedie sube un 32,95 % del entrenamiento al *holdout* (de 0,1259 a 0,1674), muy por encima de la tolerancia del 5 %, y las métricas de escala confirman el deterioro: el MAE sube un 12,0 % (de 93,3 a 104,5), el RMSE un 12,7 % (de 135,3 a 152,5) y el SMAPE un 15,1 %. Junto con los residuos de entrenamiento sin estructura, esto indica que el modelo de 120 minutos se ajusta mejor a la ventana de entrenamiento de lo que generaliza: según el protocolo, hay evidencia de sobreajuste. Aun así, el MASE sigue por debajo de 1 (sube de 0,516 a 0,662): en el *holdout*, el MAE del modelo es de 104,5 pasajeros por ventana de dos horas frente a 157,9 del ingenuo (33,8 % menos), con una demanda media de 824 pasajeros. El sesgo pasa de −7,8 en entrenamiento a +39,1 en el *holdout* (sobreestima un 4,7 % de la demanda media).

In [34]:
comparar_metricas(final_h8, preds_h8)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,0.1259,0.1674,+32.9503%
MAE,93.3049,104.5468,+12.0485%
RMSE,135.3346,152.5043,+12.6868%
MSE,18315.4512,23257.5490,+26.9832%
Bias,-7.7564,39.1457,+604.6867%
SMAPE,0.1195,0.1375,+15.0771%
MASE,0.5160,0.6622,+28.3491%
Pinball (τ=0.50),46.6524,52.2734,+12.0485%


Holdout | demanda media: 824.3 pasajeros por ventana de 120 min | MAE modelo: 104.5 | MAE ingenuo: 157.9 | mejora: 33.8 %


Ajuste en el *holdout*: demanda observada frente a predicha. Los totales diarios del panel superior equivalen aproximadamente a 8 veces los pasajeros del día; los paneles inferiores muestran pasajeros por ventana de 120 minutos en la primera semana de septiembre y el 15 de octubre de 2025.

El modelo sobreestima en todos los meses, más al inicio del periodo: 9,6 % en junio, 7,4 % en julio y 5,7 % en agosto, bajando a 0,7 % en noviembre y subiendo a 5,4 % en diciembre. Los días con mayor sobreestimación son el 25 y el 24 de diciembre (63 % y 59 %), el 19 de junio (43 %) y tres domingos seguidos de un lunes festivo: el 12 de octubre (34 %), el 16 de noviembre (29 %) y el 7 de diciembre (31 %), además del 8 de diciembre (28 %). En los acercamientos, el 15 de octubre predice cerca de 1.520 pasajeros en el pico de la mañana frente a unos 1.830 observados, y sigue bien el resto del día.

In [35]:
grafica_ajuste(
    pd.to_datetime(preds_h8["date"]), preds_h8["y_true"].to_numpy(), preds_h8["y_pred"].to_numpy(), 8,
    ZOOMS_HOLDOUT, "Demanda acumulada de pasajeros Holdout 2025 (120 min)",
)

## Conclusiones

Los tres modelos LightGBM-Tweedie de la ruta Calle 72 superan al modelo ingenuo semanal en el *holdout* (26 de mayo a 31 de diciembre de 2025): el MAE es un 31,5 % menor en 30 minutos, un 32,6 % menor en 60 minutos y un 33,8 % menor en 120 minutos, y el MASE queda por debajo de 1 en los tres (0,685; 0,674 y 0,662).

| Horizonte | Potencia $p$ | Demanda media por ventana | MAE modelo | MAE ingenuo | MAE / demanda media | Cambio de la desviación Tweedie | Sesgo en *holdout* |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 30 min | 1,1122 | 206 | 50,3 | 73,4 | 24,4 % | −0,68 % | +11,0 |
| 60 min | 1,5227 | 412 | 70,3 | 104,4 | 17,1 % | −8,25 % | +17,7 |
| 120 min | 1,8448 | 824 | 104,5 | 157,9 | 12,7 % | +32,95 % | +39,1 |

**Precisión relativa.** El error en proporción a la demanda baja al aumentar el horizonte: 24,4 % en 30 minutos, 17,1 % en 60 y 12,7 % en 120, porque al acumular más franjas las variaciones de corto plazo se compensan. Las desviaciones Tweedie no se comparan entre horizontes, porque cada uno usa su propia potencia y escala.

**Generalización.** Solo el modelo de 30 minutos cumple la tolerancia del 5 % del protocolo. El de 60 minutos la supera en la dirección de un menor error (−8,25 %), con un cambio del MAE de solo −3,7 %, por lo que no hay indicios de sobreajuste. El de 120 minutos es el resultado más débil: la desviación sube un 32,95 % y el MAE un 12,0 % fuera de muestra, lo que indica sobreajuste; aun así, sigue superando al ingenuo por un tercio.

**Sesgo y días atípicos.** Los tres modelos sobreestiman en el *holdout* entre un 4,3 % y un 5,3 % de la demanda media. La sobreestimación se concentra en días con demanda atípicamente baja que el calendario del modelo no distingue: el 24, el 25 y el 31 de diciembre, el 19 de junio y el 29 de julio, y en 120 minutos también los domingos previos a un lunes festivo (12 de octubre, 16 de noviembre y 7 de diciembre). Esto explica también que el error del *holdout* conserve autocorrelación: en esos días el modelo se equivoca en la misma dirección durante muchas ventanas seguidas.

**Picos de demanda.** En los tres horizontes el modelo reproduce la forma del día, pero queda por debajo de los picos de la mañana en días puntuales: el 15 de octubre de 2025 predice cerca del 66 %, el 82 % y el 83 % del pico observado en 30, 60 y 120 minutos.

**Potencia Tweedie.** Las tres potencias se estimaron con el bloque de calentamiento y ninguna quedó en el límite del recorte. Crecen con el horizonte (1,11; 1,52 y 1,84): al acumular más franjas, la distribución pasa de estar cerca de una Poisson a acercarse a una Gamma.

**Ventana de historia.** Los modelos de 30 y 120 minutos eligieron 22.000 franjas (unos 11 meses), dentro del espacio de búsqueda. El de 60 minutos eligió 26.000, el máximo permitido, por lo que en ese horizonte conviene probar ventanas más largas.

**Uso para la programación de frecuencias.** Los modelos sirven como base para los días hábiles y fines de semana regulares. En vísperas de festivo, domingos de puente y fin de año conviene ajustar a la baja la demanda pronosticada, y en la hora pico de la mañana conviene dejar margen hacia arriba. Las dos mejoras pendientes son una variable de calendario que marque esos días y una búsqueda con más regularización para el horizonte de 120 minutos.